In [1]:
import json
import os, sys
import pickle
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn import metrics
import seaborn as sns
import ROOT as rt


## user define
sys.path.append(".")
sys.path.append("./Utils")
from Plotter import HistStack
from CommonTrain import *
from Model import Seq

2023-05-04 01:10:44.299328: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6cb32/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-90caf/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorfl

Welcome to JupyROOT 6.26/08


# Set config

In [2]:
setting={}
with open('config/Settrain.json') as json_file:
    data = json.load(json_file)
    setting=data
INPUTDIR = setting["INPUTDIR"]
VERSION = setting["VERSION"]
# SIGNAL_MASSES = ["Zp-1700_CH-345", "Zp-2900_CH-345", "Zp-4100_CH-345", "Zp-2900_CH-1095", "Zp-4100_CH-1095", "Zp-4100_CH-1845"]

# ERA="ee17"
print(VERSION)
INPUTDIR = sys.argv[1]
REGION = sys.argv[2]
PART = sys.argv[3]

# # for dev
# CHANNEL="em"
# VERSION = '20230430wp80_ANv3-3'
# INPUTDIR = setting["INPUTDIR"]+'/em18/'
# # INPUTDIR = '/eos/user/h/hkwon/Run2/Preselection/20230430wp80_ANv3/Presel/ee17/'
# # INPUTDIR = '/eos/user/h/hkwon/SWAN_projects/Zpeak/20230502pdf_17/Presel/ee17/'
# REGION = 'SR'
# PART = 'dummy'

20230430wp80_ANv3


# Load data

In [3]:
if REGION == 'SR':
    file = 'nested_dic_df_SR1.pkl'
if REGION == 'VR':   
    file = 'nested_dic_df_TTCR.pkl'
#     file = 'nested_dic_df_PT.pkl'
# file = "/"+ERA+"/nested_dic_df.pkl"
with open(INPUTDIR+'/'+file, 'rb') as f:
    nested_dic_df = pickle.load(f)
 
dic_df=nested_dic_df['raw']
dic_df_met_u_unsort=nested_dic_df['met_u']
dic_df_met_d_unsort=nested_dic_df['met_d'] 

# # temp!!
for key in dic_df_met_u_unsort:
    df = dic_df_met_u_unsort[key]
#     print(df[df.isin([np.inf, -np.inf]).any(1)])
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        dic_df_met_u_unsort[key] = df.drop(i)
    df = dic_df_met_d_unsort[key]
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        dic_df_met_d_unsort[key] = df.drop(i)
        
selected_columns = [
#                     'MC1C1_pre', 'H3PP_pre', 'H3TPP_pre', 
                    'pt_lead_pre', 'mass_pre', # by auc rank
                    'MT_pre', 'pt_trail_pre', 'Zpt_pre', 'dRll_pre', 'PuppiMET_pre',
                    'dPhilMET_pre', 'dPhillMET_pre', 'U_pre', 'dPhill_pre', 'MT2_pre',
                    'JZB_pre'] # order used in training

# # dic_df_meta={}
# # check b jet
# query_str="nbjet_pre==0"

# print("####################################################")
# print("####   (unweighted) #events after preselection  ####")
# print("####################################################")
# for key in dic_df:  
# #     dic_df_meta[key]=dic_df[key].query(query_str)
#     dic_df[key]=dic_df[key].query(query_str) #temp!
#     print(key, ":", dic_df[key].shape[0])
    
dic_df_train={}    
dic_df_met_u={}
dic_df_met_d={}
for key in dic_df:
    print(key)
    #dic_df_train[key]=pd.concat([dic_df[key][selected_columns], dic_df1[key][selected_columns]]) #when use ll channel
    #dic_df[key]=pd.concat([dic_df[key], dic_df1[key]])
#     dic_df_train[key]=dic_df_meta[key][selected_columns] # check b jet   
    dic_df_train[key]=dic_df[key][selected_columns]    
    dic_df_met_u[key]=dic_df_met_u_unsort[key][selected_columns]    
    dic_df_met_d[key]=dic_df_met_d_unsort[key][selected_columns]

Warning, nan in TT
Int64Index([216870], dtype='int64')
Warning, nan in TT
Int64Index([216870], dtype='int64')
Warning, nan in TTUp
Int64Index([95379], dtype='int64')
DY1000To1500
DY1500To2000
DY2000To3000
DY500To700
DY700To800
DY800To1000
Zp-1700_CH-345
Zp-1700_CH-595
Zp-1700_CH-845
Zp-2100_CH-345
Zp-2100_CH-595
Zp-2100_CH-845
Zp-2500_CH-345
Zp-2500_CH-595
Zp-2500_CH-845
Zp-2900_CH-1095
Zp-2900_CH-1345
Zp-2900_CH-345
Zp-2900_CH-595
Zp-2900_CH-845
Zp-3300_CH-1095
Zp-3300_CH-1345
Zp-3300_CH-1595
Zp-3300_CH-345
Zp-3300_CH-595
Zp-3300_CH-845
Zp-3700_CH-1095
Zp-3700_CH-1345
Zp-3700_CH-1595
Zp-3700_CH-1845
Zp-3700_CH-345
Zp-3700_CH-595
Zp-3700_CH-845
Zp-4100_CH-1095
Zp-4100_CH-1345
Zp-4100_CH-1595
Zp-4100_CH-1845
Zp-4100_CH-345
Zp-4100_CH-595
Zp-4100_CH-845
TT
TThdampUp
TThdampDown
TTUp
TTDown
TTSemi
TTZ
DY
top
antitop
ZZ
WZ
WW
Data


In [4]:
print(dic_df["Zp-2500_CH-345"].columns)
print(dic_df_train["Zp-2500_CH-345"].columns)
print(dic_df_met_u["Zp-2500_CH-345"].columns)

Index(['PuppiMET_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'MET_pre', 'MET_phi_pre',
       'PuppiMET_phi_pre', 'eta_lead_pre', 'eta_trail_pre', 'phi_lead_pre',
       'phi_trail_pre', 'njet_pre', 'nbjetflav_pre', 'genweight_u_pre',
       'genweight_d_pre', 'genweight_pre', 'puweight_pre', 'puweight_u_pre',
       'puweight_d_pre', 'l1prefiring_pre', 'l1prefiring_u_pre',
       'l1prefiring_d_pre', 'effSF_pre', 'effSF_u_pre', 'effSF_d_pre',
       'trigSF_pre', 'trigSF_u_pre', 'trigSF_d_pre', 'topSF_pre',
       'topSF_u_pre', 'topSF_d_pre', 'btagSF_pre', 'pt_mu_pre', 'pt_el_pre',
       'eta_mu_pre', 'eta_el_pre', 'phi_mu_pre', 'phi_el_pre', 'weight',
       'wopu', 'wol1', 'woeff', 'wotop', 'wobsf', 'weighthdampUp',
       'weighthdampDown', 'weightPSUp', 'weightPSDown', 'weightpuweightUp',
       'weightpuweightDown', 'weightl1prefiringUp', 'weightl1pr

In [5]:
def get_x(sig_key, dic_df_train):
    # x: sig and all bkgs concatenated
    x = dic_df_train[sig_key].copy(deep=True)
    for k in dic_df_train:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_df_train[k]], axis=0)
    
    return x

In [6]:
def get_xy(sig_key, dic_predict_sort):
    # x: sig and all bkgs concatenated
    x = dic_predict_sort[sig_key].copy(deep=True)
    for k in dic_predict_sort:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_predict_sort[k]], axis=0)
            
    tot_sig = dic_predict_sort[sig_key].shape[0]
    tot_bkgs = x.shape[0] - tot_sig        
    y_sig = np.full((tot_sig,1), 1)
    y_bkgs = np.full((tot_bkgs,1), 0) 
    y = np.concatenate((y_sig, y_bkgs), axis=0) 
    
    return x, y

In [7]:
import math
# sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "met", "trigSF"]
sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "met", "trigSF", "hdamp", "PS"]

# sys_list = ["l1prefiring"]
if REGION == 'SR':
    threshold = 0.6
else:
    threshold = 1    
    
def draw_with_systematics(model_mass, dic_df_raw, dic_nested_shape_sys, sys_list, variable, r, region):
    dic_sys = {}
    dic_sys_sqr = {}
    # nominal weight plot
    file_prefix = "Outputs/"+REGION
    hist=HistStack(dic_df_raw, variable, 'weight', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/norm_dnnscore.pdf", blind_from=threshold)     
    dic_yields = hist.get_yields_by_bin()
    bkg_norm = dic_yields['bkg']
    hist.create_root_file(file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    
    for s in sys_list:
        if "met" in s or "scale" in s: # fill later
            continue
        # up
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Up', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systUp_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysUp = dic_yields['bkg']
        dic_sys[s+'Up'] = np.array(sysUp) - np.array(bkg_norm)
        hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        # down
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Down', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systDown_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysDown = dic_yields['bkg']
        dic_sys[s+'Down'] = np.array(sysDown) - np.array(bkg_norm)
        hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        
    # shape sys
    hist=HistStack(dic_nested_shape_sys['met_u'], variable, 'weightmetUp', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist("Zp-2500_CH-345", ERA, file_name=file_prefix+"/sys/systUp_"+s+".pdf")     
    dic_yields = hist.get_yields_by_bin()
    sysUp = dic_yields['bkg']
    dic_sys['metUp'] = np.array(sysUp) - np.array(bkg_norm)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    hist=HistStack(dic_nested_shape_sys['met_d'], variable, 'weightmetDown', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist("Zp-2500_CH-345", ERA, file_name=file_prefix+"/sys/systDown_"+s+".pdf")     
    dic_yields = hist.get_yields_by_bin()
    sysUp = dic_yields['bkg']
    dic_sys['metDown'] = np.array(sysUp) - np.array(bkg_norm)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        
    # si^2
    for s in sys_list:
        # by source
        sqrUp = [x*x if x>0 else -x*x for x in dic_sys[s+'Up']]
        sqrUp_up = [x if x>0 else 0 for x in sqrUp]
        sqrUp_down = [x if x<0 else 0 for x in sqrUp]
        sqrDown = [x*x if x>0 else -x*x for x in dic_sys[s+'Down']]
        sqrDown_up = [x if x>0 else 0 for x in sqrDown]
        sqrDown_down = [x if x<0 else 0 for x in sqrDown]
        
        sqr_up = np.array(sqrUp_up) + np.array(sqrDown_up)
        sqr_down = np.array(sqrUp_down) + np.array(sqrDown_down)
        
        dic_sys_sqr[s+'Up'] = sqr_up
        dic_sys_sqr[s+'Down'] = np.fabs(sqr_down) #store absolute
        
    print(dic_sys)
    print(dic_sys_sqr)
    # sqrt(sumi si^2)
    for i, s in enumerate(sys_list):
        if i==0:
            sum_up = dic_sys_sqr[s+'Up'].copy() # be carefull not to use shallow copy
            sum_down = dic_sys_sqr[s+'Down'].copy()
        else:
            sum_up += dic_sys_sqr[s+'Up']
            sum_down += dic_sys_sqr[s+'Down']

    dic_sys_sqr['sum_up'] = np.sqrt(sum_up)
#     print(sum_up)
#     print(sum_down)
    dic_sys_sqr['sum_down'] = np.sqrt(sum_down)
    print(dic_sys_sqr)
            
    return dic_sys_sqr

In [8]:
from multiprocessing import Pool
def wrapper_mp(model_mass):
    
#     # dump roc
#     dic_roc = {}
#     x, y = get_xy(model_mass, dic_predict_sort)
#     tpr, fpr, auc = dump_roc(model_mass, y, x, dic_predict_sort)
#     dic_roc['tpr'] = tpr
#     dic_roc['fpr'] = fpr
#     dic_roc['auc'] = auc
#     dic_nested_roc[model_mass] = dic_roc            


    #-- implement systematics --#   
    os.system("mkdir -p Outputs/"+REGION)
    os.system("mkdir -p Outputs/"+REGION+"/sys")
#     sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "trigSF"]
    # sys_list = ["puweight"]

    r = [nbin, 0, 1, "DNN score"]

    if REGION == 'SR':
        threshold = 0.6
    else:
        threshold = 1    
    
    # draw plot with syst
    dic_sys_sqr = draw_with_systematics(model_mass, dic_predict_sort, dic_nested_shape_sys, sys_list, "score"+model_mass, r, "region_dummy")

    hist=HistStack(dic_predict_sort, "score"+model_mass, 'weight', nbin, 0, 1, "DNN score", dic_sys=dic_sys_sqr) 
    
    c1=hist.draw_hist(model_mass, ERA, file_name="Outputs/"+REGION+"/"+model_mass+VERSION, blind_from=threshold)     
    hist.get_yeilds()
    hist.draw_pie(file_name="Outputs/"+REGION+"/"+model_mass+VERSION)     
    hist.export_yields_table(file_name="Outputs/"+REGION+"/"+model_mass)     
     
    return None

# Load model and evaluate

In [9]:
import time

nbin = 20
if REGION == "VR":
    nbin = 10
# VERSION = '0903'
# VERSION = setting["VERSION"]
print(time.time())

one_mass = 'Zp-2500_CH-345'

if 'mm' in INPUTDIR:
    CHANNEL = 'mm'
if 'ee' in INPUTDIR:
    CHANNEL = 'ee'
if 'em' in INPUTDIR:
    CHANNEL = 'em'
    
OUTDIR = 'Outputs/'+CHANNEL
file_postfix = 'PNN_'+VERSION
# file_postfix = one_mass+VERSION

# model_masses = ['Zp-1700_CH-345', 'Zp-2500_CH-345', 'Zp-2900_CH-345', 'Zp-3300_CH-345', 'Zp-4100_CH-345']
model_masses = [m for m in dic_df_train if 'Zp' in m] # if want to run all
# if PART=="one":
#     model_masses = [m for m in dic_df_train if 'Zp-3' in m or 'Zp-4' in m]
# if PART=="two":
#     model_masses = [m for m in dic_df_train if 'Zp-1' in m or 'Zp-2' in m]
# model_masses = ['Zp-2500_CH-345', 'Zp-4100_CH-345']

# %matplotlib inline
rt.gROOT.SetBatch(rt.kTRUE)

dic_nested_roc = {}

ERA = INPUTDIR

model = keras.models.load_model(OUTDIR+'/model_'+file_postfix)
#     model = keras.models.load_model(OUTDIR+'/model_binary_'+file_postfix)

Transformer = pickle.load(open(OUTDIR+'/scaler_'+file_postfix+'.pkl','rb'))

nested_dic_predict, nested_dic_predict_met_u, nested_dic_predict_met_d = [{} for _ in range(3)]

# masses loop
for model_mass in model_masses:
    
    masses = model_mass.split('_')
    zp_m = masses[0].split('-')
    zp_m = zp_m[1]
    ch_m = masses[1].split('-')
    ch_m = ch_m[1]

#     file_postfix = 'test'

    dic_df_scale, dic_df_scale_met_u, dic_df_scale_met_d = [{} for _ in range(3)]        

    for key2 in dic_df_train:
        dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
        dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
        dic_df_met_u[key2]['zp_m'] = np.full(dic_df_met_u[key2].shape[0], int(zp_m))
        dic_df_met_u[key2]['ch_m'] = np.full(dic_df_met_u[key2].shape[0], int(ch_m))
        dic_df_met_d[key2]['zp_m'] = np.full(dic_df_met_d[key2].shape[0], int(zp_m))
        dic_df_met_d[key2]['ch_m'] = np.full(dic_df_met_d[key2].shape[0], int(ch_m))
        if dic_df_train[key2].empty:
            continue
        dic_df_scale[key2] = Transformer.transform(dic_df_train[key2]) 
        dic_df_scale_met_u[key2] = Transformer.transform(dic_df_met_u[key2]) 
        dic_df_scale_met_d[key2] = Transformer.transform(dic_df_met_d[key2]) 

    # predict
    dic_predict, dic_predict_met_u, dic_predict_met_d = [{} for _ in range(3)]

    for key3 in dic_df_scale:
    #             if "Zp-" in key3 and not key in key3: # use same scaling used in model training for all masses
    #                 continue
        dic_predict[key3] = model.predict(dic_df_scale[key3], batch_size=dic_df_scale[key3].shape[0])
        dic_predict_met_u[key3] = model.predict(dic_df_scale_met_u[key3], batch_size=dic_df_scale_met_u[key3].shape[0])
        dic_predict_met_d[key3] = model.predict(dic_df_scale_met_d[key3], batch_size=dic_df_scale_met_d[key3].shape[0])
    
    nested_dic_predict[model_mass] = dic_predict
    nested_dic_predict_met_u[model_mass] = dic_predict_met_u
    nested_dic_predict_met_d[model_mass] = dic_predict_met_d
    

# array in y-axis to dataframe
dic_predict_sort, dic_predict_met_u_sort, dic_predict_met_d_sort = [{} for _ in range(3)]

for key4 in dic_df_scale:
    new_df = dic_df[key4].copy(deep=True).reset_index(drop="True")
    new_df_met_u = pd.DataFrame()
    new_df_met_d = pd.DataFrame()
    new_df_met_u["weightmetUp"] = dic_df_met_u_unsort[key4]["weightmetUp"].copy(deep=True).reset_index(drop="True")
    new_df_met_d["weightmetDown"] = dic_df_met_d_unsort[key4]["weightmetDown"].copy(deep=True).reset_index(drop="True")    
    dic_predict_sort[key4] = new_df
    dic_predict_met_u_sort[key4] = new_df_met_u
    dic_predict_met_d_sort[key4] = new_df_met_d
    
for model_mass in model_masses:
    for key5 in dic_df_scale:
        dic_predict_sort[key5]["score"+model_mass] = nested_dic_predict[model_mass][key5][:,0]
        dic_predict_met_u_sort[key5]["score"+model_mass] = nested_dic_predict_met_u[model_mass][key5][:,0]
        dic_predict_met_d_sort[key5]["score"+model_mass] = nested_dic_predict_met_d[model_mass][key5][:,0]
    
dic_nested_shape_sys={}
dic_nested_shape_sys["met_u"]=dic_predict_met_u_sort
dic_nested_shape_sys["met_d"]=dic_predict_met_d_sort         
with Pool() as p:
    p.map(wrapper_mp, model_masses)  


with open(OUTDIR+'/nested_dic_roc_'+file_postfix+'.pkl', 'wb') as f:
    pickle.dump(dic_nested_roc, f)            
        


1683155469.4260547


/tmp/ipykernel_6872/2436076586.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_6872/2436076586.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_6872/2436076586.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

/tmp/ipykernel_6872/2436076586.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_6872/2436076586.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_6872/2436076586.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

/tmp/ipykernel_6872/2436076586.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_6872/2436076586.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_6872/2436076586.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

/tmp/ipykernel_6872/2436076586.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_6872/2436076586.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_met_u[key2]['zp_m'] = np.full(dic_df_met_u[key2].shape[0], int(zp_m))
/tmp/ipykernel_6872/2436076586.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

...

          3.53107548,    5.06229349,    4.4722748 ,    2.02703526])}
puweightUp
puweightDown
l1prefiringUp
l1prefiringDown
effSFUp
effSFDown
topSFUp
topSFDown
metUp
metDown
trigSFUp
trigSFDown
hdampUp
hdampDown
PSUp
PSDown
sum_up
sum_down
['puweight', 'l1prefiring', 'effSF', 'topSF', 'met', 'trigSF', 'hdamp', 'PS']
array('f', [27.690000534057617, 0.09500519931316376, 31.21328353881836, 1082.904052734375, 2431.375732421875, 23426.525390625, 23788.1484375, 23220.4609375, 23516.052734375, 23435.705078125, 68.05069732666016, 48.018341064453125])
test
puweightUp
puweightDown
l1prefiringUp
l1prefiringDown
effSFUp
effSFDown
topSFUp
topSFDown
metUp
metDown
trigSFUp
trigSFDown
hdampUp
hdampDown
PSUp
PSDown
sum_up
sum_down
['puweight', 'l1prefiring', 'effSF', 'topSF', 'met', 'trigSF', 'hdamp', 'PS']
array('f', [27.690000534057617, 0.09500519931316376, 31.21328353881836, 1082.904052734375, 2431.375732421875, 23426.525390625, 23788.1484375, 23220.4609375, 23516.052734375, 23435.705078125, 68.05069

       1.22866279e+00, 0.00000000e+00, 1.40487232e+01, 0.00000000e+00])}
{'puweightUp': array([1.98837030e+03, 6.78948973e+01, 4.88125498e+00, 1.89737090e+00,
       1.62051538e+00, 2.35015089e-02, 4.09492309e-01, 1.07612515e-01,
       1.26346218e+00, 7.13240802e-02, 6.29793504e-01, 8.08145680e-01,
       2.97642726e-02, 2.27087049e-01, 2.49462731e-02, 4.11445452e-01,
       1.11881500e-02, 4.87924699e-01, 1.05989114e-01, 9.34672506e-02]), 'puweightDown': array([2.32109213e+03, 5.39018958e+01, 4.14394957e+00, 2.55732023e+00,
       2.32849636e+00, 1.04336168e-01, 2.50283097e+00, 6.24335768e-05,
       9.94008728e-01, 1.44117916e-03, 1.18524905e+00, 3.77414966e-01,
       5.08665717e-02, 6.39443668e-01, 1.28691237e-01, 3.18441009e-01,
       5.56772533e-03, 5.40834448e-01, 1.63103029e-01, 2.06191860e-03]), 'l1prefiringUp': array([3.48710917e+02, 7.48144412e-01, 3.49850856e-01, 1.31396858e-01,
       6.87349129e-02, 3.78870834e-02, 5.10173538e-02, 3.85316061e-02,
       2.46794420e-02, 

        1.30304135e+00, -6.17849200e-01, -1.77471155e-01,  2.15955378e-01])}
{'puweightUp': array([1.58839567e+03, 8.01519162e+01, 2.89154011e+01, 2.62723912e+00,
       2.10490225e-02, 5.06281304e-02, 1.41249403e+00, 1.88413567e-01,
       6.26249466e-01, 3.58952246e+00, 9.79018051e-02, 6.04040799e-01,
       1.93039269e-03, 0.00000000e+00, 5.71051874e-01, 2.18767127e-03,
       4.28647267e-01, 8.64416754e-02, 1.82250320e-01, 1.56994921e-01]), 'puweightDown': array([1.96109862e+03, 5.66922420e+01, 2.61767604e+01, 4.25377258e+00,
       6.60100534e-02, 1.83745788e-01, 2.85286789e+00, 6.72223173e-02,
       3.45609809e-01, 2.19235340e+00, 3.07979262e-01, 6.78884105e-01,
       3.30669935e-02, 9.65156509e-02, 5.50556280e-01, 3.24944682e-03,
       3.94351332e-01, 1.25440294e-01, 2.15249629e-01, 1.19237617e-02]), 'l1prefiringUp': array([3.44705290e+02, 9.98509964e-01, 4.06894311e-01, 1.35787693e-01,
       6.00332723e-02, 3.33214802e-02, 4.62212544e-02, 4.07133644e-02,
       2.75641498e-

          2.51411869,    2.50296119,    5.61397713,    1.83210163])}
puweightUp
puweightDown
l1prefiringUp
l1prefiringDown
effSFUp
effSFDown
topSFUp
topSFDown
metUp
metDown
trigSFUp
trigSFDown
hdampUp
hdampDown
PSUp
PSDown
sum_up
sum_down
['puweight', 'l1prefiring', 'effSF', 'topSF', 'met', 'trigSF', 'hdamp', 'PS']
array('f', [27.690000534057617, 0.09500519931316376, 31.21328353881836, 1082.904052734375, 2431.375732421875, 23426.525390625, 23788.1484375, 23220.4609375, 23516.052734375, 23435.705078125, 68.05069732666016, 48.018341064453125])
test
puweightUp
puweightDown
l1prefiringUp
l1prefiringDown
effSFUp
effSFDown
topSFUp
topSFDown
metUp
metDown
trigSFUp
trigSFDown
hdampUp
hdampDown
PSUp
PSDown
sum_up
sum_down
['puweight', 'l1prefiring', 'effSF', 'topSF', 'met', 'trigSF', 'hdamp', 'PS']
array('f', [27.690000534057617, 0.09500519931316376, 31.21328353881836, 1082.904052734375, 2431.375732421875, 23426.525390625, 23788.1484375, 23220.4609375, 23516.052734375, 23435.705078125, 68.05069

2023-05-04 01:11:10.146308: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6cb32/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-90caf/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow/contrib

# signal unc.

In [10]:
def write_sys_hist(h1, h2):
    file_name = "Outputs/"+REGION+"/shape_v1_"+VERSION+model_mass
    f = rt.TFile(file_name+"_v1.root","UPDATE") 
    f.cd("signal_region")
#     h1.Write("zp_"+sys+"Up", rt.TObject.kWriteDelete)
#     h2.Write("zp_"+sys+"Down", rt.TObject.kWriteDelete)
    h1.Write()
    h2.Write()  

In [11]:
from collections import defaultdict
import math

c = rt.TCanvas()
c.SetGrid()
rt.gStyle.SetOptStat(0)
    
# def export_pdf_scale_unc(model_mass): 
for model_mass in model_masses:    
    w0 = dic_predict_sort[model_mass]['weight'].copy(deep=True)
#     hists = []
#     for j in range(31):
#         h = rt.TH1D("", "", nbin, 0, 1)
#         for i, v in enumerate(dic_predict_sort[model_mass]["score"+model_mass]):
#             w = w0[i]*dic_predict_sort[model_mass]['pdf'+str(j)+'_pre'][i]
#             h.Fill(v, w)
#     #     h.Draw("same")
#         hists.append(h)
#     # print(hists)
# #     c.Draw()
# #     c.SaveAs("Syst/pdfs_"+model_mass+".pdf")

#     dic_yields=defaultdict(list)
#     for i, h in enumerate(hists):
#         for nb in range(nbin):
#             dic_yields["pdf"+str(i)].append(h.GetBinContent(nb+1))

#     unc_pdfs = []
#     for nb in range(nbin):
#         detla_sqrs = []
#         for i in range(30):
#             delta = dic_yields["pdf"+str(i+1)][nb] - dic_yields["pdf0"][nb]
#             detla_sqr = delta*delta
#             detla_sqrs.append(detla_sqr)
#         sum_delta_sqrs = sum(detla_sqrs)
#         unc_pdf = math.sqrt(sum_delta_sqrs)
#         unc_pdfs.append(unc_pdf)

#     hist=HistStack(dic_predict_sort, "score"+model_mass, 'weight', nbin, 0, 1, "DNN score") 
#     c1=hist.draw_hist(model_mass, ERA, file_name="Outputs/"+REGION+"/normpdf"+model_mass)
#     # print(hist.get_yields_by_bin()[model_mass])

#     pdf_norm = np.array(hist.get_yields_by_bin()[model_mass])
#     # weight
#     unc_pdfs1 = np.array(unc_pdfs)*pdf_norm[-1]/dic_yields["pdf0"][-1]

#     print("norm:", pdf_norm[-1], dic_yields["pdf0"][-1], "new:", unc_pdfs1[-1], unc_pdfs[-1])
    
#     pdf_up = pdf_norm + unc_pdfs1
#     pdf_down = pdf_norm - unc_pdfs1

#     h_pdfup = rt.TH1D("zp_pdfUp", "zp_pdfUp", nbin, 0, 1)
#     h_pdfdown = rt.TH1D("zp_pdfDown", "zp_pdfDown", nbin, 0, 1)
#     for nb in range(nbin):
#         h_pdfup.SetBinContent(nb+1, pdf_up[nb])
#         h_pdfdown.SetBinContent(nb+1, pdf_down[nb])

    h_scaleUp = rt.TH1D("zp_scaleUp", "zp_scaleUp", nbin, 0, 1)
    h_scaleDown = rt.TH1D("zp_scaleDown", "zp_scaleDown", nbin, 0, 1)
    for i, v in enumerate(dic_predict_sort[model_mass]["score"+model_mass]):
        wup = w0[i]*dic_predict_sort[model_mass]['genweight_u_pre'][i]
        wdn = w0[i]*dic_predict_sort[model_mass]['genweight_d_pre'][i]
        h_scaleUp.Fill(v, wup)
        h_scaleDown.Fill(v, wdn)

    write_sys_hist(h_scaleUp, h_scaleDown)
#     write_sys_hist(h_pdfup, h_pdfdown)



Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleUp (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleDown (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleUp (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleDown (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleUp (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleDown (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleUp (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleDown (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleUp (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleDown (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: zp_scaleUp (Potential memory leak).
Warning in <TROOT::App